# Thực nghiệm MNN – chạy lại riêng tập IBTrACS 6h với cấu hình huấn luyện giảm dao động

Tách từ `MNN_ThucNghiem_3TapDuLieu.ipynb`, **chỉ chạy tập `IBTrACS_6h`** (`ibtracs_tbd_6h.csv`). Mô hình, danh sách cấu hình (ABL1, ABL2, LEAK, BENCH, TWO), cách chia tập và đánh giá **giữ nguyên**; chỉ đổi quy trình huấn luyện để đường loss bớt dao động:

| Tham số | Bản cũ | Bản này | Tác dụng |
|---|---|---|---|
| Learning rate ban đầu (Adam) | 1e-3 | **5e-4** | bước cập nhật nhỏ hơn, bớt nhảy quanh điểm tối ưu |
| Giảm learning rate khi chững lại | không | **có**: `LR_PATIENCE` = 5 epoch không cải thiện → lr × 0,5, tối thiểu 1e-5 | áp dụng cho **mọi** cấu hình (cả RMSProp) |
| Batch size | 256 | **512** | gradient bớt nhiễu |

Kết quả lưu ở thư mục riêng `KetQua_IBTrACS6h_giamdaodong/`, không lẫn với kết quả cũ. Mỗi lần chạy lưu thêm lịch sử learning rate (`lr_hist`) để vẽ cùng đường hội tụ.

Các tập dữ liệu khác của bản gốc (để tham khảo):

| Khoá | File | Vị trí | Speed/dir "có sẵn" |
|---|---|---|---|
| `IBTrACS_6h` | `ibtracs_tbd_6h.csv` | LAT, LON | STORM_SPEED / STORM_DIR của IBTrACS |
| `IBTrACS_CARQ` | `ibtracs__carq.csv` | LAT, LON | STORM_SPEED / STORM_DIR của IBTrACS |
| `CARQ_NHC` | `carq_wide_clean.csv` | `lat_best`, `lon_best` (best track) | `spd_0` / `dir_0` của NHC (thời gian thực) |

Mỗi đặc trưng speed/dir có **hai bản**:
- **tự tính**: sai phân lùi từ vị trí t−6h → t (chỉ dùng quá khứ);
- **có sẵn**: lấy thẳng từ dữ liệu (IBTrACS: nghi rò rỉ; CARQ: giá trị NHC xác định lúc phát tin).

| Phần | Nội dung |
|---|---|
| 1 | Đọc 3 tập, cắt phân đoạn, chia 70/15/15 theo cơn bão và theo thời gian |
| 2 | Kiểm toán: speed/dir có sẵn khớp với sai phân lùi / trung tâm / tiến bao nhiêu % |
| 3–5 | Tạo cửa sổ, các mô hình (MNN song tuyến tính, M-neuron, MLP, LSTM, GRU, TCN, Transformer, hồi quy tuyến tính), huấn luyện |
| 6 | Danh sách cấu hình: **ABL1** (thêm riêng từng thành phần vào MNN gốc), **ABL2** (cộng dồn, mẫu của cô), **LEAK**, **BENCH**, **TWO** (hai giai đoạn) |
| 7 | Chạy toàn bộ (5 seed × 3 tầm × 3 tập), **chạy tiếp được** nếu bị ngắt |
| 8 | Bảng kết quả: mean ± std theo seed, skill so với persistence, so sánh cặp có CI95 bootstrap theo cơn bão, bảng gộp 3 tập |
| 9 | Phân tích theo nhóm (lưu vực, đổi hướng, giai đoạn, cường độ), đường hội tụ một/hai giai đoạn |

Kết quả từng lần chạy lưu ở `KetQua_3TapDuLieu/runs/`. Chạy lại notebook sẽ **bỏ qua** những lần đã có; phần 8–9 dùng mọi kết quả đã có (kể cả khi mới xong một phần seed).

## Thư viện

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'   # tránh crash kernel do trùng thư viện OpenMP trên Windows

import re
import time
import json
import random
import shutil

import numpy as np
import tensorflow as tf2              # import TF trước matplotlib/sklearn
import tensorflow.compat.v1 as tf
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

tf.disable_v2_behavior()
tf2.get_logger().setLevel('ERROR')

try:
    from IPython.display import display
except ImportError:                    # chạy ngoài Jupyter
    display = print

plt.rc('font', **{'size': 11})
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 80)

for name, mod in [('numpy', np), ('pandas', pd), ('matplotlib', matplotlib),
                  ('sklearn', sklearn), ('tensorflow', tf2)]:
    print(f'{name:11s} {mod.__version__}')

## 0. Cấu hình chung

- `RUN_DATASETS`, `RUN_TABLES`, `SEEDS`, `FORECAST_LIST`: chọn phần cần chạy.
- Thứ tự chạy: **hết seed đầu cho mọi tập / mọi cấu hình rồi mới sang seed sau**, để sớm có một bộ kết quả đầy đủ.
- `OPT_PRESETS`: `rmsprop` là cách tối ưu của MNN gốc (RMSProp, lr 1e-4, L2 0,02 trên W, V); `adam` là của mô hình đề xuất (Adam, **lr 5e-4**, cắt gradient ±1, L2 5e-4).
- **Giảm dao động**: `LR_PATIENCE`, `LR_FACTOR`, `MIN_LR` (giảm lr khi val chững lại) và `BATCH_SIZE = 512`. Với `STOP_PATIENCE = 30`, lr có thể giảm nhiều lần trước khi dừng sớm.
- `QUICK = True` chỉ để thử đường ống (2 epoch, 1 seed, ít cơn bão); số liệu khi đó **không có ý nghĩa** và được lưu vào thư mục riêng.

In [ ]:
SEED = 42                                   # seed cho phần không phải huấn luyện (bootstrap...)

DATASETS = {
    'IBTrACS_6h':   {'path': 'ibtracs_tbd_6h.csv',  'fmt': 'ibtracs', 'prov': 'STORM_SPEED/DIR (IBTrACS)'},
    'IBTrACS_CARQ': {'path': 'ibtracs__carq.csv',   'fmt': 'ibtracs', 'prov': 'STORM_SPEED/DIR (IBTrACS)'},
    'CARQ_NHC':     {'path': 'carq_wide_clean.csv', 'fmt': 'carq',    'prov': 'spd_0/dir_0 (NHC CARQ)'},
}
RUN_DATASETS = ['IBTrACS_6h']               # notebook này chỉ chạy lại tập IBTrACS 6h
RUN_TABLES = ['ABL1', 'ABL2', 'LEAK', 'BENCH', 'TWO']

FORECAST_LIST = [1, 2, 4]                   # số bước 6h: +6h, +12h, +24h
WINDOW_SIZE = 4
FIRST_HID = [2, 25, 8]
SECOND_HID = [2, 50, 10]
SEEDS = [20, 42, 123, 300, 2024]

MAX_EPOCHS = 200
STOP_PATIENCE = 30
MIN_REL_IMPROVE = 2e-3                      # val phải giảm > 0,2% mới tính là cải thiện
BATCH_SIZE = 512                            # bản cũ: 256
OPT_PRESETS = {
    'rmsprop': {'name': 'RMSProp', 'lr': 1e-4, 'l2': 0.02, 'clip': None},   # MNN gốc (Training.py ban đầu)
    'adam':    {'name': 'Adam',    'lr': 5e-4, 'l2': 5e-4, 'clip': 1.0},    # mô hình đề xuất (bản cũ: lr 1e-3)
}
LR_PATIENCE = 5                             # số epoch liền val không giảm quá 0,2% thì giảm lr
LR_FACTOR = 0.5                             # mỗi lần giảm: lr × 0,5
MIN_LR = 1e-5                               # lr không xuống dưới mức này
BASE_HID = 64                               # kích thước ẩn của LSTM / GRU / TCN; Transformer d_model = 32

MAX_STEP_KM = 600.0                         # cắt phân đoạn khi một bước 6h > 600 km
MIN_SEG_LEN = WINDOW_SIZE + max(FORECAST_LIST) + 1
KM_PER_DEG = 111.195
TOL_DIR, TOL_SPD = 10.0, 2.0                # ngưỡng "khớp" khi kiểm toán speed/dir (độ, kt)
TURN_DEG = 45.0                             # đổi hướng: góc giữa bước 6h vừa qua và độ dời thật > 45°
MIN_MOVE_KM = 10.0
N_BOOT = 2000

QUICK = False
QUICK_STORMS = 150
OUT_DIR = 'KetQua_IBTrACS6h_giamdaodong'
if QUICK:
    MAX_EPOCHS, SEEDS, OUT_DIR = 2, SEEDS[:1], 'KetQua_IBTrACS6h_giamdaodong_QUICK'
RUN_DIR = os.path.join(OUT_DIR, 'runs')
os.makedirs(RUN_DIR, exist_ok=True)


def find_file(p):
    for q in [p, os.path.join('..', p)]:
        if os.path.exists(q):
            return q
    raise FileNotFoundError(f"Không tìm thấy '{p}' ở thư mục notebook hoặc thư mục cha")


for k in RUN_DATASETS:
    DATASETS[k]['path'] = find_file(DATASETS[k]['path'])
    print(f'{k:13s} -> {os.path.abspath(DATASETS[k]["path"])}')
random.seed(SEED)
np.random.seed(SEED)

## 1. Dữ liệu

Mỗi tập được đưa về cùng một dạng rồi cắt phân đoạn như `MNN_Main.py`: cắt ở chỗ hụt mốc 6h hoặc chỗ nhảy hơn `MAX_STEP_KM`; bỏ phân đoạn ngắn hơn `MIN_SEG_LEN` điểm (= 9, đủ cho cả +24h). Mọi tầm dự báo dùng **cùng một tập phân đoạn**.

Mỗi phân đoạn thành một bảng 8 cột:

| Cột | 0 | 1 | 2, 3, 4 | 5, 6, 7 |
|---|---|---|---|---|
| Nội dung | kinh độ (liên tục, không nhảy qua 180°) | vĩ độ | speed (kt), sin, cos hướng **tự tính** t−6h → t | speed, sin, cos hướng **có sẵn** |

Hướng đo từ Bắc theo chiều kim đồng hồ (hướng bão đi tới). Hàng 0 của mỗi phân đoạn không bao giờ nằm trong cửa sổ đầu vào, nên không cần giá trị tự tính ở đó.

In [ ]:
def haversine_distance(coords_true, coords_pred):
    """Khoảng cách Haversine (km); mỗi hàng là (lon, lat) theo độ."""
    R = 6371.0
    lon1, lat1 = np.radians(coords_true[:, 0]), np.radians(coords_true[:, 1])
    lon2, lat2 = np.radians(coords_pred[:, 0]), np.radians(coords_pred[:, 1])
    a = (np.sin((lat2 - lat1) / 2.0) ** 2 +
         np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2.0) ** 2)
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def wrap180(x):
    return (x + 180.0) % 360.0 - 180.0


def motion(lon0, lat0, lon1, lat1, hours):
    """Tốc độ (kt) và hướng (độ) đi từ điểm 0 tới điểm 1, hệ phẳng địa phương."""
    latm = np.radians((lat0 + lat1) / 2.0)
    dx = wrap180(lon1 - lon0) * KM_PER_DEG * np.cos(latm)
    dy = (lat1 - lat0) * KM_PER_DEG
    return np.hypot(dx, dy) / hours / 1.852, np.degrees(np.arctan2(dx, dy)) % 360.0


def read_dataset(key):
    """Đọc file và đưa về các cột chung: sid, time, lat, lon, spd_p, dir_p, name, year, basin, nature, vmax."""
    cfg = DATASETS[key]
    if cfg['fmt'] == 'ibtracs':
        df = pd.read_csv(cfg['path'], low_memory=False, keep_default_na=False, na_values=[''])
        d = pd.DataFrame({'sid': df['ID'].astype(str), 'time': pd.to_datetime(df['ISO_TIME'], errors='coerce'),
                          'lat': pd.to_numeric(df['LAT'], errors='coerce'),
                          'lon': pd.to_numeric(df['LON'], errors='coerce'),
                          'spd_p': pd.to_numeric(df['STORM_SPEED'], errors='coerce'),
                          'dir_p': pd.to_numeric(df['STORM_DIR'], errors='coerce'),
                          'name': df['NAME'].astype(str), 'year': pd.to_numeric(df['YEAR'], errors='coerce'),
                          'basin': df['BASIN'].astype(str), 'nature': df['NATURE'].astype(str), 'vmax': np.nan})
    else:   # carq_wide_clean.csv
        df = pd.read_csv(cfg['path'], low_memory=False, dtype={'cy': str, 'cycle_dtg': str})
        d = pd.DataFrame({'sid': df['storm_id'].astype(str),
                          'time': pd.to_datetime(df['cycle_dtg'], format='%Y%m%d%H', errors='coerce'),
                          'lat': df['lat_best'].astype(float), 'lon': df['lon_best'].astype(float),
                          'spd_p': df['spd_0'].astype(float), 'dir_p': df['dir_0'].astype(float),
                          'name': df['stormname'].fillna('UNNAMED').astype(str), 'year': df['year'],
                          'basin': df['basin'].astype(str).str.upper(), 'nature': df['ty_0'].fillna('XX').astype(str),
                          'vmax': pd.to_numeric(df['vmax_0'], errors='coerce')})
    d = d.dropna(subset=['time', 'lat', 'lon']).sort_values(['sid', 'time'])
    d = d.drop_duplicates(['sid', 'time'])
    d[['spd_p', 'dir_p']] = d.groupby('sid', sort=False)[['spd_p', 'dir_p']].ffill().fillna(0.0)
    if QUICK:
        first = d.groupby('sid')['time'].min().sort_values()
        d = d[d['sid'].isin(first.index[:QUICK_STORMS])]
    return d


def build_segments(d, step_hours=6):
    tables, meta = [], []
    n_jump = 0
    for sid, g in d.groupby('sid', sort=False):
        lon, lat = g['lon'].to_numpy(float), g['lat'].to_numpy(float)
        dt_h = g['time'].diff().dt.total_seconds().to_numpy() / 3600.0
        step = np.r_[0.0, haversine_distance(np.c_[lon[:-1], lat[:-1]], np.c_[lon[1:], lat[1:]])]
        gap, jump = dt_h[1:] != step_hours, step[1:] > MAX_STEP_KM
        n_jump += int(np.sum(jump & ~gap))
        for seg_no, idx in enumerate(np.split(np.arange(len(g)), np.where(gap | jump)[0] + 1)):
            if len(idx) < MIN_SEG_LEN:
                continue
            s = g.iloc[idx]
            s_lat = lat[idx]
            s_lon = (lon[idx][0] % 360.0) + np.r_[0.0, np.cumsum(wrap180(np.diff(lon[idx])))]   # kinh độ liên tục
            spd_s, dir_s = np.zeros(len(idx)), np.zeros(len(idx))
            spd_s[1:], dir_s[1:] = motion(s_lon[:-1], s_lat[:-1], s_lon[1:], s_lat[1:], step_hours)
            spd_s[0], dir_s[0] = spd_s[1], dir_s[1]
            rs, rp = np.radians(dir_s), np.radians(s['dir_p'].to_numpy(float))
            t = np.column_stack([s_lon, s_lat, spd_s, np.sin(rs), np.cos(rs),
                                 s['spd_p'].to_numpy(float), np.sin(rp), np.cos(rp)])
            tables.append(t)
            meta.append({'id': sid, 'seg': seg_no, 'name': str(s['name'].iloc[0]).strip(),
                         'year': int(s['year'].iloc[0]), 'start': s['time'].iloc[0],
                         'basin': s['basin'].to_numpy(), 'nature': s['nature'].to_numpy(),
                         'vmax': s['vmax'].to_numpy(float), 'dir_p': s['dir_p'].to_numpy(float)})
    return tables, meta, n_jump


def split_by_storm(meta, f_train=0.7, f_val=0.85):
    """Chia theo cơn bão, theo thứ tự thời điểm bắt đầu cơn bão (70/15/15)."""
    start = {}
    for m in meta:
        start[m['id']] = min(start.get(m['id'], m['start']), m['start'])
    ids = sorted(start, key=lambda k: (start[k], k))
    a, b = int(len(ids) * f_train), int(len(ids) * f_val)
    sets = [set(ids[:a]), set(ids[a:b]), set(ids[b:])]
    return [[i for i, m in enumerate(meta) if m['id'] in s] for s in sets]


DATA = {}
for key in RUN_DATASETS:
    d = read_dataset(key)
    tables, meta, n_jump = build_segments(d)
    DATA[key] = {'tables': tables, 'meta': meta, 'split': split_by_storm(meta), 'n_rows': len(d),
                 'n_storms_raw': d['sid'].nunique(), 'n_jump': n_jump}
    print(f'{key:13s}: {len(d):,} dòng, {d["sid"].nunique():,} cơn bão -> {len(tables):,} phân đoạn '
          f'từ {len({m["id"] for m in meta}):,} cơn ({n_jump} chỗ cắt do nhảy > {MAX_STEP_KM:.0f} km)')

### 1.1 Thống kê chia tập

Số cơn bão, phân đoạn, cửa sổ từng tầm và ranh giới năm của mỗi tập; kiểm tra không có cơn bão nào nằm ở hai tập. Dùng để điền Bảng IV, V và bảng quy trình ATCF trong báo cáo.

In [ ]:
SPLIT_NAMES = ['Huấn luyện', 'Kiểm định', 'Kiểm thử']
rows = []
for key in RUN_DATASETS:
    T, M, S = DATA[key]['tables'], DATA[key]['meta'], DATA[key]['split']
    id_sets = [{M[i]['id'] for i in idx} for idx in S]
    for name, idx, ids in zip(SPLIT_NAMES, S, id_sets):
        starts = [M[i]['start'] for i in idx]
        rows.append({'Tập dữ liệu': key, 'Tập': name, 'Số cơn bão': len(ids), 'Số phân đoạn': len(idx),
                     'Số điểm': sum(T[i].shape[0] for i in idx),
                     **{f'Cửa sổ +{6 * fs}h': sum(max(T[i].shape[0] - WINDOW_SIZE - fs, 0) for i in idx)
                        for fs in FORECAST_LIST},
                     'Bắt đầu từ': min(starts).strftime('%d/%m/%Y'), 'đến': max(starts).strftime('%d/%m/%Y')})
    overlap = sum(len(id_sets[a] & id_sets[b]) for a in range(3) for b in range(a + 1, 3))
    print(f'{key}: số cơn bão nằm ở hơn một tập = {overlap}')
split_stats = pd.DataFrame(rows)
split_stats.to_csv(os.path.join(OUT_DIR, 'Thong_ke_chia_tap.csv'), index=False, encoding='utf-8-sig')
display(split_stats)

## 2. Kiểm toán speed/dir "có sẵn"

So speed/dir có sẵn tại mốc *t* với speed/dir tính từ vị trí theo 4 cách: **lùi 6h** (t−6 → t), **lùi 12h** (t−12 → t), **trung tâm** (t−6 → t+6) và **tiến 6h** (t → t+6). Khớp khi lệch hướng ≤ `TOL_DIR` và lệch tốc độ ≤ `TOL_SPD`; bỏ các điểm có tốc độ có sẵn bằng 0.

Nguồn nào khớp **trung tâm / tiến** tốt hơn **lùi** là đã dùng vị trí sau thời điểm *t* (rò rỉ).

In [ ]:
def audit_dataset(key):
    refs = {'Lùi 6h (t−6→t)': (-1, 0, 6), 'Lùi 12h (t−12→t)': (-2, 0, 12),
            'Trung tâm (t−6→t+6)': (-1, 1, 12), 'Tiến 6h (t→t+6)': (0, 1, 6)}
    acc = {k: ([], []) for k in refs}
    for t, m in zip(DATA[key]['tables'], DATA[key]['meta']):
        n = t.shape[0]
        sp_p, dr_p = t[:, 5], m['dir_p']
        for k, (a, b, h) in refs.items():
            r = np.arange(max(0, -a), n - max(0, b))
            if len(r) == 0:
                continue
            sp, dr = motion(t[r + a, 0], t[r + a, 1], t[r + b, 0], t[r + b, 1], h)
            ok = sp_p[r] > 0
            acc[k][0].append(np.abs(wrap180(dr_p[r] - dr))[ok])
            acc[k][1].append(np.abs(sp_p[r] - sp)[ok])
    rows = []
    for k in refs:
        ed, es = np.concatenate(acc[k][0]), np.concatenate(acc[k][1])
        rows.append({'Tập dữ liệu': key, 'Nguồn': DATASETS[key]['prov'], 'So với': k, 'Số điểm': len(ed),
                     'Lệch hướng trung vị (°)': np.median(ed), 'Lệch tốc độ trung vị (kt)': np.median(es),
                     f'% khớp (≤{TOL_DIR:g}°, ≤{TOL_SPD:g} kt)': 100 * np.mean((ed <= TOL_DIR) & (es <= TOL_SPD))})
    return rows


audit = pd.DataFrame([r for key in RUN_DATASETS for r in audit_dataset(key)])
audit.round(2).to_csv(os.path.join(OUT_DIR, 'Kiem_toan_speed_dir.csv'), index=False, encoding='utf-8-sig')
display(audit.round(2))
display(audit.pivot(index='So với', columns='Tập dữ liệu', values=audit.columns[-1]).round(1))

## 3. Tạo cửa sổ đầu vào

Tên đặc trưng:

| Tên | Ý nghĩa |
|---|---|
| `lon`, `lat` | vị trí tuyệt đối (độ) – đầu vào của **MNN gốc** |
| `dx`, `dy` | độ dời km của từng bước 6h (quy theo vĩ độ điểm neo) |
| `spd_s`, `sin_s`, `cos_s` | speed / hướng **tự tính** |
| `spd_p`, `sin_p`, `cos_p` | speed / hướng **có sẵn** |

**Đầu vào có `lon`/`lat` thì mục tiêu là vị trí tuyệt đối tại t+h** (như MNN gốc); các trường hợp khác mục tiêu là độ dời km từ điểm neo. Mọi cấu hình dùng cùng các điểm neo nên **các điểm đánh giá giống hệt nhau**.

In [ ]:
FEAT_COL = {'lon': 0, 'lat': 1, 'spd_s': 2, 'sin_s': 3, 'cos_s': 4, 'spd_p': 5, 'sin_p': 6, 'cos_p': 7}


def is_abs(feats):
    return 'lon' in feats or 'lat' in feats


def anchor_indices(n_rows, forecast_steps):
    return np.arange(WINDOW_SIZE, n_rows - forecast_steps)


def windows_one(t, fs, feats):
    js = anchor_indices(t.shape[0], fs)
    if len(js) == 0:
        return np.zeros((0, WINDOW_SIZE, len(feats))), np.zeros((0, 1, 2))
    rows = js[:, None] + np.arange(-WINDOW_SIZE + 1, 1)[None, :]          # (N, W)
    cos_ref = np.cos(np.radians(t[js, 1]))[:, None]
    X = np.empty((len(js), WINDOW_SIZE, len(feats)))
    for q, f in enumerate(feats):
        if f == 'dx':
            X[:, :, q] = (t[rows, 0] - t[rows - 1, 0]) * KM_PER_DEG * cos_ref
        elif f == 'dy':
            X[:, :, q] = (t[rows, 1] - t[rows - 1, 1]) * KM_PER_DEG
        else:
            X[:, :, q] = t[rows, FEAT_COL[f]]
    if is_abs(feats):
        Y = t[js + fs, :2]
    else:
        tgt = t[js + fs, :2] - t[js, :2]
        Y = np.c_[tgt[:, 0] * KM_PER_DEG * cos_ref[:, 0], tgt[:, 1] * KM_PER_DEG]
    return X, Y[:, None, :]


_WIN_CACHE = {}


def get_windows(key, part, fs, feats):
    """part: 0 = train, 1 = val, 2 = test. Nối theo đúng thứ tự phân đoạn."""
    ck = (key, part, fs, tuple(feats))
    if ck not in _WIN_CACHE:
        T = DATA[key]['tables']
        XY = [windows_one(T[i], fs, feats) for i in DATA[key]['split'][part]]
        _WIN_CACHE[ck] = (np.concatenate([a for a, _ in XY]), np.concatenate([b for _, b in XY]))
    return _WIN_CACHE[ck]


_EVAL_CACHE = {}


def eval_info(key, part, fs):
    """Thông tin để đổi dự báo về toạ độ và để phân nhóm, cho từng điểm đánh giá."""
    ck = (key, part, fs)
    if ck in _EVAL_CACHE:
        return _EVAL_CACHE[ck]
    T, M = DATA[key]['tables'], DATA[key]['meta']
    out = {k: [] for k in ['anchor', 'lat_ref', 'true_abs', 'pers_km', 'storm', 'basin', 'nature', 'vmax',
                           'year', 'turn']}
    for i in DATA[key]['split'][part]:
        t, m = T[i], M[i]
        js = anchor_indices(t.shape[0], fs)
        if len(js) == 0:
            continue
        cosr = np.cos(np.radians(t[js, 1]))
        last = np.c_[(t[js, 0] - t[js - 1, 0]) * KM_PER_DEG * cosr, (t[js, 1] - t[js - 1, 1]) * KM_PER_DEG]
        true = np.c_[(t[js + fs, 0] - t[js, 0]) * KM_PER_DEG * cosr, (t[js + fs, 1] - t[js, 1]) * KM_PER_DEG]
        nl, nt = np.linalg.norm(last, axis=1), np.linalg.norm(true, axis=1)
        cosang = np.sum(last * true, axis=1) / (nl * nt + 1e-9)
        turn = np.where((nl >= MIN_MOVE_KM) & (nt >= MIN_MOVE_KM), np.degrees(np.arccos(np.clip(cosang, -1, 1))), np.nan)
        out['anchor'].append(t[js, :2]); out['lat_ref'].append(t[js, 1]); out['true_abs'].append(t[js + fs, :2])
        out['pers_km'].append(fs * last); out['storm'].append(np.repeat(m['id'], len(js)))
        out['basin'].append(m['basin'][js]); out['nature'].append(m['nature'][js]); out['vmax'].append(m['vmax'][js])
        out['year'].append(np.repeat(m['year'], len(js))); out['turn'].append(turn)
    info = {k: np.concatenate(v) for k, v in out.items()}
    info['groups'] = pd.factorize(info['storm'])[0]
    info['err_pers'] = haversine_distance(info['true_abs'], to_abs(info, info['pers_km']))
    _EVAL_CACHE[ck] = info
    return info


def to_abs(info, disp_km):
    """Độ dời km (Đông, Bắc) từ điểm neo -> (lon, lat)."""
    cosr = np.cos(np.radians(info['lat_ref']))
    return info['anchor'] + np.c_[disp_km[:, 0] / (KM_PER_DEG * cosr), disp_km[:, 1] / KM_PER_DEG]

## 4. Các mô hình

**Lớp ma trận** (`matrix_layer`), các tuỳ chọn dùng cho ablation:

| Tuỳ chọn | Giá trị | Ý nghĩa |
|---|---|---|
| `layer` | `bilinear` | song tuyến tính dùng chung: u = U·X·Vᵀ + B (như `Add_layer.py` gốc) |
| | `mneuron` | M-neuron: u[j,k] = Σ X[p,q]·W_L[j,k,p]·W_R[j,k,q] + θ[j,k] |
| `ln` | True / False | chuẩn hoá lớp ở hai lớp ẩn |
| `act` | `sigmoid` / `algebraic` | kích hoạt hai lớp ẩn; lớp ra luôn tuyến tính |
| `init` | `one` | std = 1 (như code gốc) |
| | `scaled` | std = (n₁·n₂)^(−1/4) |

**Baseline**: MLP (cửa sổ duỗi thành vector, 2 lớp ẩn, số tham số ≈ mô hình đề xuất), LSTM, GRU (64 đơn vị, trạng thái cuối → 2), TCN (tích chập nhân quả kernel 2, dilation 1 và 2), Transformer encoder (d_model 32, 2 head, 1 khối) và hồi quy tuyến tính (kiểu CLIPER). Tất cả viết bằng `tf.compat.v1`, cùng đầu vào và đầu ra.

In [ ]:
def algebraic_activation(x):
    return x / tf.sqrt(1.0 + tf.square(x))


ACTS = {'sigmoid': tf.sigmoid, 'algebraic': algebraic_activation, None: None}


def layer_norm(u, shape, axes):
    mean, var = tf.nn.moments(u, axes=axes, keepdims=True)
    gamma = tf.Variable(tf.ones(shape), name='gamma')
    beta = tf.Variable(tf.zeros(shape), name='beta')
    return gamma * (u - mean) / tf.sqrt(var + 1e-6) + beta


def matrix_layer(x, in_shape, out_shape, layer='mneuron', act=None, ln=False, init='scaled', name='layer'):
    n1, n2 = in_shape
    m1, m2 = out_shape
    s = 1.0 if init == 'one' else (n1 * n2) ** -0.25
    with tf.variable_scope(name):
        if layer == 'bilinear':
            WL = tf.Variable(tf.random.normal([m1, n1], stddev=s), name='W_L')
            WR = tf.Variable(tf.random.normal([m2, n2], stddev=s), name='W_R')
            u = tf.einsum('jp,bpq,kq->bjk', WL, x, WR)
        else:
            WL = tf.Variable(tf.random.normal([m1, m2, n1], stddev=s), name='W_L')
            WR = tf.Variable(tf.random.normal([m1, m2, n2], stddev=s), name='W_R')
            u = tf.einsum('bpq,jkp,jkq->bjk', x, WL, WR)
        u = u + tf.Variable(tf.zeros([1, m1, m2]), name='theta')
        if ln:
            u = layer_norm(u, [1, m1, m2], [1, 2])
        out = u if act is None else ACTS[act](u)
    return out, [WL, WR]


def build_mnn(xs, n_feat, arch):
    shp0, shp1, shp2 = (WINDOW_SIZE, n_feat), tuple(FIRST_HID[1:]), tuple(SECOND_HID[1:])
    l1, r1 = matrix_layer(xs, shp0, shp1, arch['layer'], arch['act'], arch['ln'], arch['init'], 'layer1')
    l2, r2 = matrix_layer(l1, shp1, shp2, arch['layer'], arch['act'], arch['ln'], arch['init'], 'layer2')
    out, r3 = matrix_layer(l2, shp2, (1, 2), arch['layer'], None, False, arch['init'], 'layer3')
    return out, r1 + r2 + r3


def dense(x, n_in, n_out, name):
    with tf.variable_scope(name):
        W = tf.Variable(tf.random.normal([n_in, n_out], stddev=np.sqrt(2.0 / (n_in + n_out))), name='kernel')
        b = tf.Variable(tf.zeros([n_out]), name='bias')
    return tf.matmul(x, W) + b, W


def full_param_count(n_feat):
    """Số tham số của mô hình đề xuất (M-neuron + LayerNorm): W_L, W_R, θ, γ, β."""
    shapes = [(WINDOW_SIZE, n_feat), tuple(FIRST_HID[1:]), tuple(SECOND_HID[1:]), (1, 2)]
    tot = 0
    for li, ((n1, n2), (m1, m2)) in enumerate(zip(shapes[:-1], shapes[1:])):
        tot += m1 * m2 * (n1 + n2) + m1 * m2 + (2 * m1 * m2 if li < 2 else 0)
    return tot


def mlp_hidden(n_feat):
    """Số đơn vị ẩn h (2 lớp ẩn bằng nhau) để MLP có số tham số gần bằng mô hình đề xuất."""
    P, n_in = full_param_count(n_feat), WINDOW_SIZE * n_feat
    return int(round((-(n_in + 4) + np.sqrt((n_in + 4) ** 2 + 4 * (P - 2))) / 2))


def build_mlp(xs, n_feat):
    h = mlp_hidden(n_feat)
    x = tf.reshape(xs, [-1, WINDOW_SIZE * n_feat])
    a1, W1 = dense(x, WINDOW_SIZE * n_feat, h, 'fc1')
    a2, W2 = dense(algebraic_activation(a1), h, h, 'fc2')
    out, W3 = dense(algebraic_activation(a2), h, 2, 'out')
    return tf.reshape(out, [-1, 1, 2]), [W1, W2, W3]


def build_lstm(xs, n_feat, h=BASE_HID):
    with tf.variable_scope('lstm'):
        Wx = tf.Variable(tf.random.normal([n_feat, 4 * h], stddev=np.sqrt(1.0 / n_feat)), name='kernel_x')
        Wh = tf.Variable(tf.random.normal([h, 4 * h], stddev=np.sqrt(1.0 / h)), name='kernel_h')
        b = tf.Variable(np.r_[np.zeros(h), np.ones(h), np.zeros(2 * h)].astype(np.float32), name='bias')
    hs = tf.zeros([tf.shape(xs)[0], h])
    c = tf.zeros([tf.shape(xs)[0], h])
    for k in range(WINDOW_SIZE):
        z = tf.matmul(xs[:, k, :], Wx) + tf.matmul(hs, Wh) + b
        i, f, g, o = tf.split(z, 4, axis=1)
        c = tf.sigmoid(f) * c + tf.sigmoid(i) * tf.tanh(g)
        hs = tf.sigmoid(o) * tf.tanh(c)
    out, Wo = dense(hs, h, 2, 'out')
    return tf.reshape(out, [-1, 1, 2]), [Wx, Wh, Wo]


def build_gru(xs, n_feat, h=BASE_HID):
    with tf.variable_scope('gru'):
        Wx = tf.Variable(tf.random.normal([n_feat, 3 * h], stddev=np.sqrt(1.0 / n_feat)), name='kernel_x')
        Wh = tf.Variable(tf.random.normal([h, 3 * h], stddev=np.sqrt(1.0 / h)), name='kernel_h')
        b = tf.Variable(tf.zeros([3 * h]), name='bias')
    hs = tf.zeros([tf.shape(xs)[0], h])
    for k in range(WINDOW_SIZE):
        gx = tf.matmul(xs[:, k, :], Wx) + b
        zx, rx, nx = tf.split(gx, 3, axis=1)
        zh, rh, nh = tf.split(tf.matmul(hs, Wh), 3, axis=1)
        z, r = tf.sigmoid(zx + zh), tf.sigmoid(rx + rh)
        n = tf.tanh(nx + r * nh)
        hs = (1.0 - z) * n + z * hs
    out, Wo = dense(hs, h, 2, 'out')
    return tf.reshape(out, [-1, 1, 2]), [Wx, Wh, Wo]


def causal_conv(x, c_in, c_out, dilation, name):
    """Tích chập nhân quả kernel 2: y_t = x_t·W0 + x_{t-d}·W1 + b."""
    with tf.variable_scope(name):
        W0 = tf.Variable(tf.random.normal([c_in, c_out], stddev=np.sqrt(2.0 / (2 * c_in))), name='kernel0')
        W1 = tf.Variable(tf.random.normal([c_in, c_out], stddev=np.sqrt(2.0 / (2 * c_in))), name='kernel1')
        b = tf.Variable(tf.zeros([c_out]), name='bias')
    x_prev = tf.pad(x, [[0, 0], [dilation, 0], [0, 0]])[:, :-dilation, :]
    return tf.einsum('btc,cd->btd', x, W0) + tf.einsum('btc,cd->btd', x_prev, W1) + b, [W0, W1]


def build_tcn(xs, n_feat, h=BASE_HID):
    y1, r1 = causal_conv(xs, n_feat, h, 1, 'tcn1')
    y2, r2 = causal_conv(tf.nn.relu(y1), h, h, 2, 'tcn2')
    y2 = tf.nn.relu(y2) + tf.nn.relu(y1)                       # nối tắt
    out, Wo = dense(y2[:, -1, :], h, 2, 'out')
    return tf.reshape(out, [-1, 1, 2]), r1 + r2 + [Wo]


def build_transformer(xs, n_feat, d=32, heads=2, ff=64):
    W = WINDOW_SIZE
    x2 = tf.reshape(xs, [-1, n_feat])
    hp, Wp = dense(x2, n_feat, d, 'proj')
    pos = tf.Variable(tf.random.normal([1, W, d], stddev=0.02), name='pos')
    h = tf.reshape(hp, [-1, W, d]) + pos
    reg = [Wp]
    with tf.variable_scope('attn'):
        Wq, Wk, Wv, Wo = [tf.Variable(tf.random.normal([d, d], stddev=np.sqrt(1.0 / d)), name=f'kernel_{n}')
                          for n in 'qkvo']
    reg += [Wq, Wk, Wv, Wo]
    dk = d // heads

    def split_heads(z):
        return tf.transpose(tf.reshape(z, [-1, W, heads, dk]), [0, 2, 1, 3])

    q, k, v = (split_heads(tf.einsum('btd,de->bte', h, M)) for M in (Wq, Wk, Wv))
    att = tf.nn.softmax(tf.matmul(q, k, transpose_b=True) / np.sqrt(dk), axis=-1)
    z = tf.reshape(tf.transpose(tf.matmul(att, v), [0, 2, 1, 3]), [-1, W, d])
    with tf.variable_scope('ln1'):
        h = layer_norm(h + tf.einsum('btd,de->bte', z, Wo), [1, 1, d], [2])
    f1, W1 = dense(tf.reshape(h, [-1, d]), d, ff, 'ff1')
    f2, W2 = dense(tf.nn.relu(f1), ff, d, 'ff2')
    with tf.variable_scope('ln2'):
        h = layer_norm(h + tf.reshape(f2, [-1, W, d]), [1, 1, d], [2])
    out, Wout = dense(h[:, -1, :], d, 2, 'out')
    return tf.reshape(out, [-1, 1, 2]), reg + [W1, W2, Wout]


def build_model(spec, xs, n_feat):
    m = spec['model']
    if m == 'mnn':
        return build_mnn(xs, n_feat, spec['arch'])
    return {'mlp': build_mlp, 'lstm': build_lstm, 'gru': build_gru, 'tcn': build_tcn,
            'transformer': build_transformer}[m](xs, n_feat)

## 5. Huấn luyện và đánh giá một cấu hình

Giống `Training.py` / `Testing.py` đã dùng: mini-batch `BATCH_SIZE`, xáo trộn mỗi epoch, z-score đầu vào và nhãn **chỉ fit trên tập huấn luyện**, lưu checkpoint tại val tốt nhất, dừng sớm khi `STOP_PATIENCE` epoch liền val không giảm quá 0,2%. Bộ tối ưu theo `OPT_PRESETS[spec['opt']]`.

**Giảm learning rate khi chững lại** (mới): đếm số epoch liền val không giảm quá 0,2% so với mốc tham chiếu; đủ `LR_PATIENCE` epoch thì lr ← max(lr × `LR_FACTOR`, `MIN_LR`) và đếm lại. Bộ đếm dừng sớm vẫn chạy độc lập. Hai giai đoạn: cập nhật luân phiên nhóm {W_L, θ, γ, β} rồi nhóm {W_R} bằng hai bộ tối ưu riêng.

Mỗi lần chạy lưu: sai số Haversine của từng điểm val/test, epoch tốt nhất, đường loss, số tham số, thời gian huấn luyện và suy luận.

In [ ]:
def run_config(key, spec, fs, seed, save_dir):
    feats = spec['feats']
    Xtr, Ytr = get_windows(key, 0, fs, feats)
    Xva, Yva = get_windows(key, 1, fs, feats)
    Xte, Yte = get_windows(key, 2, fs, feats)
    nf = Xtr.shape[2]
    sX, sY = StandardScaler().fit(Xtr.reshape(-1, nf)), StandardScaler().fit(Ytr.reshape(-1, 2))
    tx = lambda X: sX.transform(X.reshape(-1, nf)).reshape(-1, WINDOW_SIZE, nf).astype(np.float32)
    ty = lambda Y: sY.transform(Y.reshape(-1, 2)).reshape(-1, 1, 2).astype(np.float32)
    Xtr_s, Ytr_s, Xva_s, Yva_s, Xte_s = tx(Xtr), ty(Ytr), tx(Xva), ty(Yva), tx(Xte)
    t0 = time.time()
    res = {}

    if spec['model'] == 'linreg':
        reg = LinearRegression().fit(Xtr_s.reshape(len(Xtr_s), -1), Ytr_s.reshape(-1, 2))
        res.update(best_epoch=0, train_loss=np.array([]), val_loss=np.array([]), lr_hist=np.array([]),
                   n_params=reg.coef_.size + reg.intercept_.size, t_train=time.time() - t0)
        pv = reg.predict(Xva_s.reshape(len(Xva_s), -1))
        t1 = time.time()
        pt = reg.predict(Xte_s.reshape(len(Xte_s), -1))
        res['t_infer_ms'] = 1000 * (time.time() - t1) / max(len(Xte_s), 1)
    else:
        tf.reset_default_graph()
        tf.set_random_seed(seed)
        rng = np.random.RandomState(seed)
        xs = tf.placeholder(tf.float32, [None, WINDOW_SIZE, nf])
        ys = tf.placeholder(tf.float32, [None, 1, 2])
        lr_ph = tf.placeholder(tf.float32, [])
        pred, reg_vars = build_model(spec, xs, nf)
        loss = tf.reduce_mean(tf.square(ys - pred))
        opt = OPT_PRESETS[spec['opt']]
        train_loss = loss + opt['l2'] * tf.add_n([tf.nn.l2_loss(v) for v in reg_vars]) if opt['l2'] > 0 else loss

        def make_op(var_list):
            o = (tf.train.AdamOptimizer(lr_ph) if opt['name'] == 'Adam' else tf.train.RMSPropOptimizer(lr_ph))
            gv = [(g, v) for g, v in o.compute_gradients(train_loss, var_list) if g is not None]
            if opt['clip'] is not None:
                gv = [(tf.clip_by_value(g, -opt['clip'], opt['clip']), v) for g, v in gv]
            return o.apply_gradients(gv)

        all_vars = tf.trainable_variables()
        if spec.get('two_stage'):
            ops = [make_op([v for v in all_vars if 'W_R' not in v.name]),
                   make_op([v for v in all_vars if 'W_R' in v.name])]
        else:
            ops = [make_op(all_vars)]
        res['n_params'] = int(sum(np.prod(v.shape.as_list()) for v in all_vars))

        os.makedirs(save_dir, exist_ok=True)
        ckpt = os.path.join(save_dir, 'model')
        sess = tf.Session()
        sess.run(tf.global_variables_initializer())
        saver = tf.train.Saver(all_vars)

        def predict(X, chunk=8192):
            return np.concatenate([sess.run(pred, {xs: X[i:i + chunk]}) for i in range(0, len(X), chunk)])

        def full_loss(X, Y, chunk=8192):
            return sum(sess.run(loss, {xs: X[i:i + chunk], ys: Y[i:i + chunk]}) * len(X[i:i + chunk])
                       for i in range(0, len(X), chunk)) / len(X)

        trl, vll, lrh = [], [], []
        best, best_ep, ref, since = np.inf, -1, np.inf, 0
        lr, since_lr = opt['lr'], 0
        n = len(Xtr_s)
        for ep in range(MAX_EPOCHS):
            perm = rng.permutation(n)
            for i in range(0, n, BATCH_SIZE):
                b = perm[i:i + BATCH_SIZE]
                for op in ops:
                    sess.run(op, {xs: Xtr_s[b], ys: Ytr_s[b], lr_ph: lr})
            tl, vl = full_loss(Xtr_s, Ytr_s), full_loss(Xva_s, Yva_s)
            trl.append(tl); vll.append(vl); lrh.append(lr)
            if np.isfinite(vl) and vl < best:
                best, best_ep = vl, ep
                saver.save(sess, ckpt)
            if vl < ref * (1.0 - MIN_REL_IMPROVE):
                ref, since, since_lr = vl, 0, 0
            else:
                since += 1
                since_lr += 1
                if since_lr >= LR_PATIENCE and lr > MIN_LR:          # giảm lr khi val chững lại
                    lr, since_lr = max(lr * LR_FACTOR, MIN_LR), 0
                if since >= STOP_PATIENCE:
                    break
        if best_ep >= 0:
            saver.restore(sess, ckpt)
        res.update(best_epoch=best_ep, train_loss=np.array(trl), val_loss=np.array(vll), lr_hist=np.array(lrh),
                   t_train=time.time() - t0)
        pv = predict(Xva_s)
        t1 = time.time()
        pt = predict(Xte_s)
        res['t_infer_ms'] = 1000 * (time.time() - t1) / max(len(Xte_s), 1)
        sess.close()
        shutil.rmtree(save_dir, ignore_errors=True)

    for part, p in ((1, pv), (2, pt)):
        info = eval_info(key, part, fs)
        y = sY.inverse_transform(np.asarray(p).reshape(-1, 2))
        pred_abs = y if is_abs(feats) else to_abs(info, y)
        res['err_val' if part == 1 else 'err_test'] = haversine_distance(info['true_abs'], pred_abs)
    return res

## 6. Danh sách cấu hình

**ABL1 – thêm riêng từng thành phần vào MNN gốc** (không cộng dồn; mỗi hàng = MNN gốc + đúng một thay đổi).
MNN gốc: vị trí tuyệt đối (lon, lat), song tuyến tính, không LayerNorm, sigmoid, std = 1, RMSProp.

**ABL2 – cộng dồn** đúng 7 hàng theo mẫu của cô + MLP cùng số tham số. Bước cuối (Full) đổi đồng thời khởi tạo và bộ tối ưu sang của mô hình đề xuất.

**LEAK** – mô hình đề xuất với: chỉ dx, dy / + speed-dir tự tính / + speed-dir có sẵn.

**BENCH** – Persistence, hồi quy tuyến tính, LSTM, GRU, TCN, Transformer, MNN 2018, mô hình đề xuất (speed/dir tự tính).

**TWO** – mô hình đề xuất một giai đoạn và hai giai đoạn.

Hàng nào có speed/dir và nằm trong ABL1/ABL2/LEAK thì có hai bản `tự tính` và `có sẵn`. Cấu hình giống nhau giữa các bảng (vd. MNN gốc, Full tự tính) **chỉ chạy một lần** và dùng lại.

In [ ]:
ABS, DISP = ['lon', 'lat'], ['dx', 'dy']
SPD = {'tự tính': ['spd_s', 'sin_s', 'cos_s'], 'có sẵn': ['spd_p', 'sin_p', 'cos_p']}
O_ARCH = {'layer': 'bilinear', 'ln': False, 'act': 'sigmoid', 'init': 'one'}
FULL_ARCH = {'layer': 'mneuron', 'ln': True, 'act': 'algebraic', 'init': 'scaled'}


def mnn(feats, opt='rmsprop', two_stage=False, **arch):
    return {'model': 'mnn', 'feats': list(feats), 'arch': {**O_ARCH, **arch}, 'opt': opt, 'two_stage': two_stage}


def other(model, feats, opt='adam'):
    return {'model': model, 'feats': list(feats), 'opt': opt}


def full(src=None, two_stage=False):
    return mnn(DISP + (SPD[src] if src else []), 'adam', two_stage, **FULL_ARCH)


TABLES = {'ABL1': [], 'ABL2': [], 'LEAK': [], 'BENCH': [], 'TWO': []}

TABLES['ABL1'] = [
    ('MNN gốc', '—', mnn(ABS)),
    ('MNN + displacement', '—', mnn(DISP)),
    ('MNN + speed/dir', 'tự tính', mnn(ABS + SPD['tự tính'])),
    ('MNN + speed/dir', 'có sẵn', mnn(ABS + SPD['có sẵn'])),
    ('MNN + M-neuron', '—', mnn(ABS, layer='mneuron')),
    ('MNN + LayerNorm', '—', mnn(ABS, ln=True)),
    ('MNN + algebraic activation', '—', mnn(ABS, act='algebraic')),
    ('MNN + khởi tạo (n1·n2)^-1/4', '—', mnn(ABS, init='scaled')),
    ('MNN + Adam', '—', mnn(ABS, opt='adam')),
]

TABLES['ABL2'] = [('Original MNN', '—', mnn(ABS)), ('MNN + displacement', '—', mnn(DISP))]
for src in SPD:
    F = DISP + SPD[src]
    TABLES['ABL2'] += [
        ('MNN + speed/direction', src, mnn(F)),
        ('M-neuron thay bilinear', src, mnn(F, layer='mneuron')),
        ('M-neuron + LayerNorm', src, mnn(F, layer='mneuron', ln=True)),
        ('M-neuron + algebraic activation', src, mnn(F, layer='mneuron', ln=True, act='algebraic')),
        ('Full proposed model', src, full(src)),
        ('MLP cùng số tham số', src, other('mlp', F)),
    ]

TABLES['LEAK'] = [('Đề xuất: chỉ dx, dy', '—', full()),
                  ('Đề xuất + speed/dir', 'tự tính', full('tự tính')),
                  ('Đề xuất + speed/dir', 'có sẵn', full('có sẵn'))]

F_S = DISP + SPD['tự tính']
TABLES['BENCH'] = [('Hồi quy tuyến tính (kiểu CLIPER)', 'tự tính', other('linreg', F_S)),
                   ('LSTM', 'tự tính', other('lstm', F_S)),
                   ('GRU', 'tự tính', other('gru', F_S)),
                   ('TCN', 'tự tính', other('tcn', F_S)),
                   ('Transformer encoder', 'tự tính', other('transformer', F_S)),
                   ('Bilinear MNN (2018)', '—', mnn(ABS)),
                   ('Proposed M-neuron', 'tự tính', full('tự tính'))]

TABLES['TWO'] = [('Một giai đoạn', 'tự tính', full('tự tính')),
                 ('Hai giai đoạn', 'tự tính', full('tự tính', two_stage=True))]

# So sánh cặp (dòng x so với dòng y) cho bootstrap; 'Persistence' là baseline đặc biệt
COMPARE = {
    'ABL1': [(('MNN + displacement', '—'), ('MNN gốc', '—')),
             (('MNN + speed/dir', 'tự tính'), ('MNN gốc', '—')),
             (('MNN + speed/dir', 'có sẵn'), ('MNN gốc', '—')),
             (('MNN + M-neuron', '—'), ('MNN gốc', '—')),
             (('MNN + LayerNorm', '—'), ('MNN gốc', '—')),
             (('MNN + algebraic activation', '—'), ('MNN gốc', '—')),
             (('MNN + khởi tạo (n1·n2)^-1/4', '—'), ('MNN gốc', '—')),
             (('MNN + Adam', '—'), ('MNN gốc', '—'))],
    'ABL2': [(('MNN + displacement', '—'), ('Original MNN', '—'))],
    'LEAK': [(('Đề xuất + speed/dir', 'tự tính'), ('Đề xuất: chỉ dx, dy', '—')),
             (('Đề xuất + speed/dir', 'có sẵn'), ('Đề xuất + speed/dir', 'tự tính'))],
    'BENCH': [(('Proposed M-neuron', 'tự tính'), (lab, src)) for lab, src, _ in TABLES['BENCH'][:-1]]
             + [(('Proposed M-neuron', 'tự tính'), ('Persistence', '—'))],
    'TWO': [(('Hai giai đoạn', 'tự tính'), ('Một giai đoạn', 'tự tính'))],
}
for src in SPD:
    seq = [('MNN + displacement', '—')] + [(lab, src) for lab, s, _ in TABLES['ABL2'] if s == src and 'MLP' not in lab]
    COMPARE['ABL2'] += list(zip(seq[1:], seq[:-1]))
    COMPARE['ABL2'] += [(('Full proposed model', src), ('Original MNN', '—')),
                        (('MLP cùng số tham số', src), ('Full proposed model', src))]


def spec_id(spec):
    """Tên file duy nhất cho một cấu hình (cấu hình trùng giữa các bảng dùng chung kết quả)."""
    s = [spec['model'], '-'.join(spec['feats']), spec['opt']]
    if spec['model'] == 'mnn':
        a = spec['arch']
        s += [a['layer'], f"ln{int(a['ln'])}", a['act'], a['init'], f"two{int(spec['two_stage'])}"]
    return re.sub(r'[^A-Za-z0-9_.-]', '', '_'.join(s))


UNIQUE = {}
for t in RUN_TABLES:
    for lab, src, spec in TABLES[t]:
        UNIQUE.setdefault(spec_id(spec), spec)
print(f'{len(UNIQUE)} cấu hình khác nhau mỗi tập | số đặc trưng -> số đơn vị ẩn MLP: '
      f'{ {nf: mlp_hidden(nf) for nf in (5,)} } (tham số mô hình đề xuất: {full_param_count(5):,})')
for t in RUN_TABLES:
    print(f'  {t}: {len(TABLES[t])} dòng')

## 7. Chạy toàn bộ

Thứ tự: seed → tập dữ liệu → tầm dự báo → cấu hình. Mỗi lần chạy lưu một file `.npz` trong `runs/<tập>/`; nếu file đã có thì bỏ qua. Có thể dừng kernel bất cứ lúc nào rồi chạy lại ô này để tiếp tục.

In [ ]:
def run_path(key, sid, fs, seed):
    return os.path.join(RUN_DIR, key, f'{sid}_fs{fs}_s{seed}.npz')


jobs = [(seed, key, fs, sid) for seed in SEEDS for key in RUN_DATASETS for fs in FORECAST_LIST for sid in UNIQUE]
todo = [j for j in jobs if not os.path.exists(run_path(j[1], j[3], j[2], j[0]))]
print(f'Tổng {len(jobs)} lần chạy, đã có {len(jobs) - len(todo)}, còn {len(todo)}')

t_all = time.time()
for n, (seed, key, fs, sid) in enumerate(todo, 1):
    path = run_path(key, sid, fs, seed)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    print(f'[{n}/{len(todo)}] seed={seed} {key} +{6 * fs}h {sid}: ', end='', flush=True)
    res = run_config(key, UNIQUE[sid], fs, seed, os.path.join(OUT_DIR, 'ckpt', f'{key}_{sid}_{fs}_{seed}'))
    np.savez(path, **res)
    e = res['err_test']
    print(f'Mean {e.mean():7.2f} | RMSE {np.sqrt(np.mean(e ** 2)):7.2f} km | best_epoch {res["best_epoch"]} '
          f'| lr cuối {res["lr_hist"][-1] if len(res["lr_hist"]) else float("nan"):.1e} | {res["t_train"] / 60:.1f} phút')
print(f'Xong sau {(time.time() - t_all) / 3600:.2f} giờ')

## 8. Bảng kết quả

Với mỗi tập, mỗi tầm và mỗi bảng: Mean, RMSE, Median, P95 (km, tập kiểm thử) – trung bình theo các seed đã chạy ± độ lệch chuẩn theo seed; skill so với persistence; số tham số; thời gian huấn luyện (phút) và suy luận (ms/mẫu).

**So sánh cặp**: Δ = (dòng trái) − (dòng phải), âm là dòng trái tốt hơn; CI95 bootstrap **lấy mẫu theo cơn bão** (`N_BOOT` lần), mỗi lần tính chỉ số của từng seed rồi lấy trung bình.

In [ ]:
def rmse(e):
    return float(np.sqrt(np.mean(e ** 2)))


def load_errs(key, spec, fs):
    """Ma trận sai số test (số seed × số điểm) và thông tin phụ của các seed đã chạy."""
    sid = spec_id(spec)
    files = [run_path(key, sid, fs, s) for s in SEEDS if os.path.exists(run_path(key, sid, fs, s))]
    if not files:
        return None, None
    Z = [np.load(f) for f in files]
    return np.stack([z['err_test'] for z in Z]), Z


def boot_delta(Ex, Ey, groups, n_boot=N_BOOT, seed=SEED):
    """Δ Mean, Δ RMSE (x − y) và CI95, bootstrap theo cơn bão. Ex, Ey: (số seed × số điểm)."""
    G = groups.max() + 1
    cnt = np.bincount(groups, minlength=G).astype(float)
    S = lambda E, p: np.stack([np.bincount(groups, e ** p, G) for e in E])
    w = np.random.default_rng(seed).multinomial(G, np.full(G, 1.0 / G), size=n_boot).astype(float)
    nb = (w @ cnt)[:, None]
    mean_b = lambda E: ((w @ S(E, 1).T) / nb).mean(axis=1)
    rmse_b = lambda E: np.sqrt((w @ S(E, 2).T) / nb).mean(axis=1)
    dm, dr = mean_b(Ex) - mean_b(Ey), rmse_b(Ex) - rmse_b(Ey)
    pm = Ex.mean(axis=1).mean() - Ey.mean(axis=1).mean()
    pr = np.sqrt((Ex ** 2).mean(axis=1)).mean() - np.sqrt((Ey ** 2).mean(axis=1)).mean()
    return pm, np.percentile(dm, [2.5, 97.5]), pr, np.percentile(dr, [2.5, 97.5])


def fmt(m, s):
    return '—' if m is None or not np.isfinite(m) else (f'{m:.2f}' if not np.isfinite(s) else f'{m:.2f} ± {s:.2f}')


RESULTS, PAIRS = [], []
for key in RUN_DATASETS:
    for fs in FORECAST_LIST:
        info = eval_info(key, 2, fs)
        e_per = info['err_pers']
        for t in RUN_TABLES:
            E = {('Persistence', '—'): e_per[None, :]}
            rows = [{'Tập dữ liệu': key, 'Tầm (h)': 6 * fs, 'Bảng': t, 'Cấu hình': 'Persistence', 'Speed/dir': '—',
                     'n_seed': 1, 'Mean (km)': e_per.mean(), 'Std Mean': np.nan, 'RMSE (km)': rmse(e_per),
                     'Std RMSE': np.nan, 'Median (km)': np.median(e_per), 'P95 (km)': np.percentile(e_per, 95),
                     'Số tham số': 0}]
            for lab, src, spec in TABLES[t]:
                et, Z = load_errs(key, spec, fs)
                if et is None:
                    continue
                E[(lab, src)] = et
                m, r = et.mean(axis=1), np.sqrt((et ** 2).mean(axis=1))
                sd = (lambda v: v.std(ddof=1)) if len(et) > 1 else (lambda v: np.nan)
                rows.append({'Tập dữ liệu': key, 'Tầm (h)': 6 * fs, 'Bảng': t, 'Cấu hình': lab, 'Speed/dir': src,
                             'n_seed': len(et), 'Mean (km)': m.mean(), 'Std Mean': sd(m), 'RMSE (km)': r.mean(),
                             'Std RMSE': sd(r), 'Median (km)': np.median(et, axis=1).mean(),
                             'P95 (km)': np.percentile(et, 95, axis=1).mean(),
                             'Số tham số': int(Z[0]['n_params']),
                             'Best epoch': np.mean([z['best_epoch'] for z in Z]),
                             'Huấn luyện (phút)': np.mean([z['t_train'] for z in Z]) / 60,
                             'Suy luận (ms/mẫu)': np.mean([z['t_infer_ms'] for z in Z])})
            for r_ in rows:
                r_['Skill Mean (%)'] = 100 * (1 - r_['Mean (km)'] / e_per.mean())
                r_['Skill RMSE (%)'] = 100 * (1 - r_['RMSE (km)'] / rmse(e_per))
            RESULTS += rows
            for x, y in COMPARE.get(t, []):
                if x in E and y in E:
                    dm, cim, dr, cir = boot_delta(E[x], E[y], info['groups'])
                    PAIRS.append({'Tập dữ liệu': key, 'Tầm (h)': 6 * fs, 'Bảng': t,
                                  'So sánh': f'{x[0]} [{x[1]}]  so với  {y[0]} [{y[1]}]',
                                  'Δ RMSE (km)': dr, 'CI95 RMSE': f'[{cir[0]:.2f}, {cir[1]:.2f}]',
                                  'RMSE khác biệt có ý nghĩa': bool(cir[0] > 0 or cir[1] < 0),
                                  'Δ Mean (km)': dm, 'CI95 Mean': f'[{cim[0]:.2f}, {cim[1]:.2f}]',
                                  'Mean khác biệt có ý nghĩa': bool(cim[0] > 0 or cim[1] < 0)})

RES = pd.DataFrame(RESULTS)
PAIR = pd.DataFrame(PAIRS)
RES.round(4).to_csv(os.path.join(OUT_DIR, 'Ket_qua_day_du.csv'), index=False, encoding='utf-8-sig')
PAIR.round(4).to_csv(os.path.join(OUT_DIR, 'So_sanh_cap_bootstrap.csv'), index=False, encoding='utf-8-sig')
print(f'{len(RES)} dòng kết quả, {len(PAIR)} so sánh cặp -> {OUT_DIR}/')

### 8.1 Bảng gộp 3 tập (RMSE và Mean, km, mean ± std theo seed)

Mỗi bảng: hàng = cấu hình, cột = (tập dữ liệu, tầm dự báo). Lưu thành `Bang_<bảng>_RMSE.csv` và `Bang_<bảng>_Mean.csv`.

In [ ]:
def compact(table, metric):
    sub = RES[RES['Bảng'] == table].copy()
    if sub.empty:
        return pd.DataFrame()
    sub['v'] = [fmt(m, s) for m, s in zip(sub[f'{metric} (km)'], sub[f'Std {metric}'])]
    order = ['Persistence'] + list(dict.fromkeys(lab for lab, _, _ in TABLES[table]))
    sub['_o'] = (sub['Cấu hình'].map({k: i for i, k in enumerate(order)}) * 10
                 + sub['Speed/dir'].map({'—': 0, 'tự tính': 1, 'có sẵn': 2}))
    p = sub.pivot_table(index=['_o', 'Cấu hình', 'Speed/dir'], columns=['Tập dữ liệu', 'Tầm (h)'],
                        values='v', aggfunc='first')
    p = p.droplevel(0)
    return p.reindex(columns=[(k, 6 * fs) for k in RUN_DATASETS for fs in FORECAST_LIST if (k, 6 * fs) in p.columns])


TITLES = {'ABL1': 'ABL1 – thêm riêng từng thành phần vào MNN gốc', 'ABL2': 'ABL2 – cộng dồn (mẫu của cô)',
          'LEAK': 'LEAK – rò rỉ speed/dir trên mô hình đề xuất', 'BENCH': 'BENCH – so sánh với baseline',
          'TWO': 'TWO – một giai đoạn và hai giai đoạn'}
for t in RUN_TABLES:
    for metric in ['RMSE', 'Mean']:
        c = compact(t, metric)
        if c.empty:
            continue
        c.to_csv(os.path.join(OUT_DIR, f'Bang_{t}_{metric}.csv'), encoding='utf-8-sig')
        print(f'\n=== {TITLES[t]} | {metric} (km) ===')
        display(c)

### 8.2 Mức "lợi giả" của speed/dir có sẵn

So mô hình dùng speed/dir **có sẵn** với cùng mô hình dùng speed/dir **tự tính** (bảng LEAK và hàng Full của ABL2). Lợi giả (%) = 100·(RMSE tự tính − RMSE có sẵn) / RMSE tự tính. Với IBTrACS, con số lớn là dấu hiệu rò rỉ; với CARQ (speed/dir thời gian thực của NHC) kỳ vọng gần 0.

In [ ]:
rows = []
for (key, h), g in RES[RES['Bảng'] == 'LEAK'].groupby(['Tập dữ liệu', 'Tầm (h)']):
    get = lambda src: g.loc[(g['Cấu hình'] == 'Đề xuất + speed/dir') & (g['Speed/dir'] == src), 'RMSE (km)']
    s, p = get('tự tính'), get('có sẵn')
    a = g.loc[g['Cấu hình'] == 'Đề xuất: chỉ dx, dy', 'RMSE (km)']
    per = g.loc[g['Cấu hình'] == 'Persistence', 'RMSE (km)']
    if len(s) and len(p):
        rows.append({'Tập dữ liệu': key, 'Tầm (h)': h, 'Nguồn có sẵn': DATASETS[key]['prov'],
                     'RMSE persistence': per.iloc[0], 'RMSE chỉ dx,dy': a.iloc[0] if len(a) else np.nan,
                     'RMSE tự tính': s.iloc[0], 'RMSE có sẵn': p.iloc[0],
                     'Lợi giả (%)': 100 * (s.iloc[0] - p.iloc[0]) / s.iloc[0]})
leak = pd.DataFrame(rows)
if len(leak):
    leak.round(3).to_csv(os.path.join(OUT_DIR, 'Loi_gia_speed_dir.csv'), index=False, encoding='utf-8-sig')
    display(leak.round(2))
    display(leak.pivot(index='Tầm (h)', columns='Tập dữ liệu', values='Lợi giả (%)').round(1))

### 8.3 So sánh cặp (CI95 bootstrap theo cơn bão)

In [ ]:
if len(PAIR):
    for t in RUN_TABLES:
        sub = PAIR[PAIR['Bảng'] == t]
        if len(sub):
            print(f'\n=== {TITLES[t]} ===')
            display(sub.drop(columns='Bảng').set_index(['Tập dữ liệu', 'Tầm (h)', 'So sánh']).round(3))

### 8.4 Bảng benchmark đầy đủ (số tham số, thời gian)

In [ ]:
cols = ['Tập dữ liệu', 'Tầm (h)', 'Cấu hình', 'n_seed', 'Mean (km)', 'Std Mean', 'Median (km)', 'RMSE (km)',
        'Std RMSE', 'P95 (km)', 'Skill Mean (%)', 'Skill RMSE (%)', 'Số tham số', 'Huấn luyện (phút)',
        'Suy luận (ms/mẫu)']
bench = RES[RES['Bảng'] == 'BENCH']
if len(bench):
    bench[[c for c in cols if c in bench]].round(3).to_csv(os.path.join(OUT_DIR, 'Benchmark_day_du.csv'),
                                                        index=False, encoding='utf-8-sig')
    display(bench[[c for c in cols if c in bench]].round(3).set_index(['Tập dữ liệu', 'Tầm (h)', 'Cấu hình']))

### 8.5 Biểu đồ RMSE của ablation cộng dồn (ABL2)

Hàng = tập dữ liệu, cột = tầm dự báo. Thanh xanh: speed/dir tự tính; thanh cam sọc: speed/dir có sẵn; vạch đứt: persistence. Thanh lỗi: độ lệch chuẩn theo seed.

In [ ]:
sub = RES[(RES['Bảng'] == 'ABL2') & (RES['Cấu hình'] != 'Persistence')]
if len(sub):
    labels = list(dict.fromkeys(lab for lab, _, _ in TABLES['ABL2']))
    fig, axes = plt.subplots(len(RUN_DATASETS), len(FORECAST_LIST), squeeze=False,
                             figsize=(5.2 * len(FORECAST_LIST), 3.6 * len(RUN_DATASETS)))
    for r, key in enumerate(RUN_DATASETS):
        for c, fs in enumerate(FORECAST_LIST):
            ax = axes[r, c]
            g = sub[(sub['Tập dữ liệu'] == key) & (sub['Tầm (h)'] == 6 * fs)]
            x = np.arange(len(labels))
            for off, src, col, hatch in [(-0.2, 'tự tính', '#4C78A8', ''), (0.2, 'có sẵn', '#F58518', '//')]:
                vals = [g.loc[(g['Cấu hình'] == l) & (g['Speed/dir'].isin([src, '—'])), ['RMSE (km)', 'Std RMSE']]
                        for l in labels]
                v = [d['RMSE (km)'].iloc[0] if len(d) else np.nan for d in vals]
                s = [d['Std RMSE'].iloc[0] if len(d) else np.nan for d in vals]
                ax.bar(x + off, v, 0.4, yerr=np.nan_to_num(s), color=col, hatch=hatch, alpha=0.85,
                       edgecolor='white', capsize=2, label=f'speed/dir {src}')
            per = RES[(RES['Bảng'] == 'ABL2') & (RES['Cấu hình'] == 'Persistence') &
                      (RES['Tập dữ liệu'] == key) & (RES['Tầm (h)'] == 6 * fs)]['RMSE (km)']
            if len(per):
                ax.axhline(per.iloc[0], color='#555555', ls='--', lw=1.2, label='Persistence')
            ax.set_xticks(x)
            ax.set_xticklabels([f'{i + 1}' if 'MLP' not in l else 'MLP' for i, l in enumerate(labels)])
            ax.set_title(f'{key} | +{6 * fs}h')
            ax.set_ylabel('RMSE (km)')
            ax.spines[['top', 'right']].set_visible(False)
            ax.grid(axis='y', alpha=0.3)
            ax.set_axisbelow(True)
    axes[0, 0].legend(frameon=False, fontsize=9)
    fig.text(0.5, -0.01, '  '.join(f'{i + 1}: {l}' for i, l in enumerate(labels) if 'MLP' not in l), ha='center',
             fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, 'Hinh_ABL2_RMSE.png'), dpi=200, bbox_inches='tight')
    plt.show()

## 9. Phân tích theo nhóm và độ ổn định

### 9.1 Sai số theo nhóm

Cho mô hình đề xuất (speed/dir tự tính và có sẵn), MNN gốc và persistence, trên tập kiểm thử:
- **Lưu vực** (`BASIN` tại điểm neo);
- **Đi thẳng / đổi hướng**: góc giữa bước 6h vừa qua và độ dời thật tới t+h, ngưỡng `TURN_DEG`;
- **Giai đoạn thời gian**: chia năm của tập kiểm thử làm 3 khoảng;
- **Cường độ**: CARQ theo `vmax_0` (TD < 34 kt, TS 34–63 kt, HU ≥ 64 kt); IBTrACS theo `NATURE`.

In [ ]:
GROUP_MODELS = [('Đề xuất (tự tính)', full('tự tính')), ('Đề xuất (có sẵn)', full('có sẵn')),
                ('MNN gốc', mnn(ABS))]


def group_labels(key, info):
    yrs = info['year']
    edges = np.unique(np.quantile(yrs, [0, 1 / 3, 2 / 3, 1]).round())
    period = pd.cut(yrs, bins=np.r_[edges[0] - 1, edges[1:]], include_lowest=True).astype(str) if len(edges) > 1 \
        else np.repeat(str(int(yrs[0])), len(yrs))
    turn = np.where(np.isnan(info['turn']), 'gần đứng yên',
                    np.where(info['turn'] > TURN_DEG, f'đổi hướng (>{TURN_DEG:g}°)', 'đi thẳng'))
    if np.isfinite(info['vmax']).any():
        v = info['vmax']
        inten = np.where(~np.isfinite(v), 'không rõ', np.where(v < 34, 'TD (<34 kt)',
                         np.where(v < 64, 'TS (34–63 kt)', 'HU (≥64 kt)')))
    else:
        inten = info['nature']
    return {'Lưu vực': info['basin'], 'Đổi hướng': turn, 'Giai đoạn': np.asarray(period), 'Cường độ / NATURE': inten}


rows = []
for key in RUN_DATASETS:
    for fs in FORECAST_LIST:
        info = eval_info(key, 2, fs)
        errs = {'Persistence': info['err_pers']}
        for name, spec in GROUP_MODELS:
            et, _ = load_errs(key, spec, fs)
            if et is not None:
                errs[name] = et.mean(axis=0)          # trung bình theo seed tại từng điểm
        for gname, lab in group_labels(key, info).items():
            for gv in pd.unique(lab):
                m = lab == gv
                row = {'Tập dữ liệu': key, 'Tầm (h)': 6 * fs, 'Nhóm': gname, 'Giá trị': gv, 'Số điểm': int(m.sum()),
                       'Số cơn bão': len(np.unique(info['storm'][m]))}
                for name, e in errs.items():
                    row[f'Mean {name} (km)'] = e[m].mean()
                if 'Đề xuất (tự tính)' in errs:
                    row['Skill đề xuất (tự tính) vs persistence (%)'] = 100 * (1 - errs['Đề xuất (tự tính)'][m].mean()
                                                                              / errs['Persistence'][m].mean())
                rows.append(row)
grp = pd.DataFrame(rows)
grp.round(3).to_csv(os.path.join(OUT_DIR, 'Phan_tich_theo_nhom.csv'), index=False, encoding='utf-8-sig')
for gname in grp['Nhóm'].unique() if len(grp) else []:
    print(f'\n=== {gname} ===')
    display(grp[grp['Nhóm'] == gname].drop(columns='Nhóm').set_index(['Tập dữ liệu', 'Tầm (h)', 'Giá trị']).round(2))

### 9.2 Độ ổn định theo seed

Độ lệch chuẩn RMSE theo seed của từng cấu hình, gộp mọi bảng (mỗi cấu hình chỉ một dòng).

In [ ]:
stab = (RES[RES['Cấu hình'] != 'Persistence']
        .drop_duplicates(['Tập dữ liệu', 'Tầm (h)', 'Cấu hình', 'Speed/dir', 'RMSE (km)'])
        [['Tập dữ liệu', 'Tầm (h)', 'Bảng', 'Cấu hình', 'Speed/dir', 'n_seed', 'RMSE (km)', 'Std RMSE',
          'Mean (km)', 'Std Mean']])
if len(stab):
    stab['CV RMSE (%)'] = 100 * stab['Std RMSE'] / stab['RMSE (km)']
    stab.round(3).to_csv(os.path.join(OUT_DIR, 'Do_on_dinh_theo_seed.csv'), index=False, encoding='utf-8-sig')
    display(stab.round(3).set_index(['Tập dữ liệu', 'Tầm (h)', 'Cấu hình', 'Speed/dir']))

### 9.3 Đường hội tụ: một giai đoạn và hai giai đoạn

Loss train (nét mảnh) và val (nét đậm) theo epoch, trên dữ liệu đã chuẩn hoá, seed đầu tiên đã chạy. Đường chấm xám (trục phải, thang log): learning rate của cấu hình một giai đoạn.

In [ ]:
if 'TWO' in RUN_TABLES:
    fig, axes = plt.subplots(len(RUN_DATASETS), len(FORECAST_LIST), squeeze=False,
                             figsize=(5 * len(FORECAST_LIST), 3.4 * len(RUN_DATASETS)))
    for r, key in enumerate(RUN_DATASETS):
        for c, fs in enumerate(FORECAST_LIST):
            ax = axes[r, c]
            lr_drawn = False
            for (lab, _, spec), col in zip(TABLES['TWO'], ['#4C78A8', '#E45756']):
                sid = spec_id(spec)
                f = next((run_path(key, sid, fs, s) for s in SEEDS if os.path.exists(run_path(key, sid, fs, s))), None)
                if f is None:
                    continue
                z = np.load(f)
                ax.plot(z['train_loss'], color=col, lw=0.9, alpha=0.6)
                ax.plot(z['val_loss'], color=col, lw=2, label=f'{lab} (best ep {int(z["best_epoch"])})')
                if not lr_drawn and 'lr_hist' in z and len(z['lr_hist']):
                    ax2 = ax.twinx()
                    ax2.plot(z['lr_hist'], color='#888888', ls=':', lw=1.5)
                    ax2.set_yscale('log')
                    ax2.set_ylabel('learning rate', color='#888888')
                    ax2.tick_params(axis='y', colors='#888888')
                    lr_drawn = True
            ax.set_title(f'{key} | +{6 * fs}h')
            ax.set_xlabel('epoch')
            ax.set_ylabel('MSE (chuẩn hoá)')
            ax.spines[['top', 'right']].set_visible(False)
            ax.grid(alpha=0.3)
            ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, 'Hinh_hoi_tu_mot_hai_giai_doan.png'), dpi=200, bbox_inches='tight')
    plt.show()

### 9.4 Ghi chú khi đọc kết quả

- Kết luận về đóng góp của từng thành phần dựa trên các dòng **speed/dir tự tính**. Dòng **có sẵn** của IBTrACS chỉ dùng để minh hoạ rò rỉ.
- Ở CARQ, "có sẵn" là speed/dir NHC xác định lúc phát tin (không dùng vị trí tương lai); vị trí vẫn là best track.
- ABL1 giữ mọi thứ như MNN gốc (kể cả RMSProp, std = 1) và chỉ đổi một thành phần, nên dòng nào kém hơn MNN gốc nghĩa là thành phần đó **một mình** không giúp được.
- ABL2 phụ thuộc thứ tự thêm; đọc cùng ABL1 để thấy tác động riêng và tác động khi kết hợp.
- Bản này dùng lr ban đầu 5e-4, giảm lr khi chững lại và batch 512 cho **mọi** cấu hình, nên **không trộn** kết quả với bản cũ (lr 1e-3 cố định, batch 256) trong cùng một bảng.